# 06 · Qualitative crowd → image on the retrained model (all 4 aggregators)

The **visual** companion to the RQ1 numbers: crowd→image grids on the text-conditioned `diffusion_v2`, showing what each aggregator (**mean · centroid · deepsets · attention**) actually generates. These are the report's qualitative figures.

> Runtime → **GPU (T4)** — inference only. Needs `vae.pt`, `diffusion_v2/diffusion.pt`, `aggregators.pt`.

## 1. Clone & install

In [ ]:
!git clone --branch feature/crowd-driven-visual-generation https://github.com/nishant-kumar109/gen-ai-IISc.git
%cd gen-ai-IISc/projects/crowd-driven-visual-generation
!pip install -q open-clip-torch matplotlib
import torch; print('cuda', torch.cuda.is_available())

## 2. Credentials & locate checkpoints

In [ ]:
import os, getpass
from google.colab import drive; drive.mount('/content/drive')
hf_token = getpass.getpass('HF token (Enter to skip if on Drive): ').strip()
if hf_token:
    from huggingface_hub import login; login(token=hf_token)

def locate(drive_path, repo, fname):
    if os.path.exists(drive_path):
        return drive_path
    from huggingface_hub import hf_hub_download, whoami
    return hf_hub_download(f"{whoami()['name']}/{repo}", fname)

VAE_CKPT  = locate('/content/drive/MyDrive/crowdgen/vae/vae.pt', 'crowdgen-vae', 'vae.pt')
DIFF_CKPT = locate('/content/drive/MyDrive/crowdgen/diffusion_v2/diffusion.pt', 'crowdgen-diffusion-v2', 'diffusion.pt')
OUT = '/content/drive/MyDrive/crowdgen/figures_v2'; os.makedirs(OUT, exist_ok=True)
AGG = '/content/drive/MyDrive/crowdgen/diffusion_v2/aggregators.pt'
print('VAE :', VAE_CKPT); print('DIFF:', DIFF_CKPT); print('AGG :', AGG)

## 3. Themes grid — all 4 aggregators × 5 themes
Each row an aggregator, each column a theme. A crowd of ~300 words/emojis per theme → one image. Look for theme-appropriate structure, and whether the **learned** rows (deepsets/attention) look more on-theme than the heuristics.

In [ ]:
!python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} --agg-ckpt {AGG} \
    --mode themes --aggregators mean,centroid,deepsets,attention \
    --n 300 --diversity 0.2 --guidance 3.0 --steps 50 --out {OUT}/themes_4way.png
from IPython.display import Image; Image(f'{OUT}/themes_4way.png')

## 4. Diversity grid — robustness, all 4 aggregators
Same theme, rising off-theme noise. The RQ1 story *visually*: do the learned aggregators (deepsets/attention) stay on-theme as the crowd gets noisier, while mean/centroid drift?

In [ ]:
!python sample_crowd.py --vae {VAE_CKPT} --diffusion {DIFF_CKPT} --agg-ckpt {AGG} \
    --mode diversity --theme paradise --aggregators mean,centroid,deepsets,attention \
    --diversities 0.1,0.4,0.7 --n 300 --guidance 3.0 --steps 50 --out {OUT}/diversity_4way.png
from IPython.display import Image; Image(f'{OUT}/diversity_4way.png')

## Next
Figures under `MyDrive/crowdgen/figures_v2/` — the qualitative companions to the RQ1 numbers, for the report. These + the error-bar RQ1 plots are the results section. (Optional remaining: GAN baseline for a diffusion-vs-GAN comparison.)